# Phase 3: Data Cleaning and NLP Preprocessing

In [ ]:
import sys
from pathlib import Path

import pandas as pd
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    PROCESSED_DATA_DIR,
    RAW_DATA_DIR,
    dataset_overview,
    load_csv,
    save_csv,
)
from src.preprocessing import clean_text, prepare_news_dataset, prepare_sms_dataset

pd.set_option("display.max_colwidth", 120)
print("Project root:", PROJECT_ROOT)

Project root: d:\Project1\Fake-News-Spam-Detection\Fake-News-Spam-Detection


## Step 1: Load the raw datasets

In [7]:
fake_raw = load_csv(RAW_DATA_DIR / "Fake.csv")
real_raw = load_csv(RAW_DATA_DIR / "True.csv")
sms_raw = pd.read_csv("../data/raw/SMSSpamCollection",sep="\t",header=None,names=["v1", "v2"],encoding="utf-8")

print("Fake news shape:", fake_raw.shape)
print("Real news shape:", real_raw.shape)
print("SMS shape:      ", sms_raw.shape)

Fake news shape: (23481, 4)
Real news shape: (21417, 4)
SMS shape:       (5572, 2)


## Step 2: Inspect before cleaning

In [8]:
fake_overview = dataset_overview(fake_raw.assign(label="fake"), "label")
real_overview = dataset_overview(real_raw.assign(label="real"), "label")
sms_overview = dataset_overview(sms_raw, "v1")

for name, overview in [("FAKE", fake_overview), ("REAL", real_overview), ("SMS", sms_overview)]:
    print(f"--- {name} ---")
    print("rows:", overview["rows"])
    print("duplicate rows:", overview["duplicate_rows"])
    print("missing values:", {k: v for k, v in overview["missing_values"].items() if v})
    print("labels:", overview["label_counts"])
    print()

--- FAKE ---
rows: 23481
duplicate rows: 3
missing values: {}
labels: {'fake': 23481}

--- REAL ---
rows: 21417
duplicate rows: 206
missing values: {}
labels: {'real': 21417}

--- SMS ---
rows: 5572
duplicate rows: 403
missing values: {}
labels: {'ham': 4825, 'spam': 747}



## Step 3: Understand each cleaning step

In [9]:
sample = "<p>WINNER!!! Claim your £1000 prize at https://spam.example.com or mail win@promo.co.in &amp; don't wait</p>"

steps = [
    ("raw", sample),
    ("lowercase only", clean_text(sample, strip_html=False, remove_urls=False,
                                  remove_emails=False, remove_punctuation=False)),
    ("+ html removed", clean_text(sample, remove_urls=False, remove_emails=False,
                                  remove_punctuation=False)),
    ("+ urls and emails removed", clean_text(sample, remove_punctuation=False)),
    ("full clean_text", clean_text(sample)),
]
for label, value in steps:
    print(f"{label:<28} -> {value}")

raw                          -> <p>WINNER!!! Claim your £1000 prize at https://spam.example.com or mail win@promo.co.in &amp; don't wait</p>
lowercase only               -> <p>winner!!! claim your £1000 prize at https://spam.example.com or mail win@promo.co.in &amp; don't wait</p>
+ html removed               -> winner!!! claim your £1000 prize at https://spam.example.com or mail win@promo.co.in & don't wait
+ urls and emails removed    -> winner!!! claim your £1000 prize at or mail & don't wait
full clean_text              -> winner claim your 1000 prize at or mail dont wait


## Step 4: Build the cleaned news dataset

In [10]:
news_clean, news_stats = prepare_news_dataset(fake_raw, real_raw)

print("Stats:", news_stats)
print()
print(news_clean["label"].value_counts())
print()
news_clean.head()

Stats: {'input_rows': 44898, 'dropped_empty': 9, 'dropped_duplicates': 5814, 'output_rows': 39075}

label
real    21175
fake    17900
Name: count, dtype: int64



,text,label
0,donald trump sends out embarrassing new years eve message this is disturbing donald trump just couldn t wish all ame...,fake
1,drunk bragging trump staffer started russian collusion investigation house intelligence committee chairman devin nun...,fake
2,sheriff david clarke becomes an internet joke for threatening to poke people in the eye on friday it was revealed th...,fake
3,trump is so obsessed he even has obamas name coded into his website images on christmas day donald trump announced t...,fake
4,pope francis just called out donald trump during his christmas speech pope francis used his annual christmas day mes...,fake


## Step 5: Build the cleaned SMS dataset

In [11]:
sms_clean, sms_stats = prepare_sms_dataset(sms_raw)

print("Stats:", sms_stats)
print()
print(sms_clean["label"].value_counts())
print()
sms_clean.head()

Stats: {'input_rows': 5572, 'dropped_empty': 2, 'dropped_duplicates': 446, 'output_rows': 5124}

label
ham     4498
spam     626
Name: count, dtype: int64



,text,label
0,go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat,ham
1,ok lar joking wif u oni,ham
2,free entry in 2 a wkly comp to win fa cup final tkts 21st may 2005 text fa to 87121 to receive entry question std tx...,spam
3,u dun say so early hor u c already then say,ham
4,nah i dont think he goes to usf he lives around here though,ham


## Step 6: Check the result

In [12]:
import re

def validate_clean(df, name):
    problems = {
        "uppercase": int(df["text"].str.contains(r"[A-Z]", regex=True).sum()),
        "html tags": int(df["text"].str.contains(r"<[^>]+>", regex=True).sum()),
        "urls": int(df["text"].str.contains(r"https?://|www\.", regex=True).sum()),
        "empty": int((df["text"].str.len() == 0).sum()),
        "null text": int(df["text"].isna().sum()),
    }
    print(f"{name}: {problems}")
    assert all(count == 0 for count in problems.values()), f"{name} still has problems"

validate_clean(news_clean, "news")
validate_clean(sms_clean, "sms")
print("Validation passed.")

news: {'uppercase': 0, 'html tags': 0, 'urls': 0, 'empty': 0, 'null text': 0}
sms: {'uppercase': 0, 'html tags': 0, 'urls': 0, 'empty': 0, 'null text': 0}
Validation passed.


## Step 7: Save the cleaned datasets

In [13]:
news_path = save_csv(news_clean, PROCESSED_DATA_DIR / "news_clean.csv")
sms_path = save_csv(sms_clean, PROCESSED_DATA_DIR / "sms_clean.csv")
print("Saved:", news_path)
print("Saved:", sms_path)

Saved: D:\Project1\Fake-News-Spam-Detection\Fake-News-Spam-Detection\data\processed\news_clean.csv
Saved: D:\Project1\Fake-News-Spam-Detection\Fake-News-Spam-Detection\data\processed\sms_clean.csv


## Stopwords and stemming: